# RF Diffusion Implementation
https://github.com/RosettaCommons/RFdiffusion

Newer cards may need: https://github.com/RosettaCommons/RFdiffusion/issues/349
1. conda create -n my_cuda_env python=3.11
2. conda activate my_cuda_env
3. conda install -c "nvidia/label/cuda-12.8.0" cuda-toolkit
4. pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
5. conda install -c dglteam/label/th24_cu124 dgl
6. Install SE3Transformer as per RFdiffusion
7. pip install pandas
8. Add `weights_only=False` argument to `_Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))` in "~/miniconda3/envs/my_cuda_env/lib/python3.11/site-packages/e3nn/o3/_wigner.py"

## Setup

### Accept Terms of Service

In [1]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/msys2

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
accepted Terms of Service for https://repo.anaconda.com/pkgs/msys2


### Create Conda environment

In [2]:
!conda env create -f ../Tools/RFdiffusion/env/SE3nv.yml

2 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
 - pytorch
 - dglteam
 - nvidia
Platform: linux-64
Solving environment: done

cudatoolkit-11.1.1   | 929.6 MB  |                                       |   0% 
dgl-cuda11.1-0.9.1po | 223.5 MB  |                                       |   0% 

mkl-2021.4.0         | 142.6 MB  |                                       |   0% 


pytorch-1.9.1        | 45.2 MB   |                                       |   0% 



scipy-1.10.1         | 23.1 MB   |                                       |   0% 




python-3.9.24        | 23.1 MB   |                                       |   0% 





torchvision-0.15.2   | 9.8 MB    |                                       |   0% 






numpy-base-1.24.3    | 6.9 MB    |                                       |   0% 







torchaudio-0.9.1     | 4.4 MB    |                                       |   0% 








intel-openmp-2021.4. | 4.2 MB    |                                       |   0% 





Please select the SE3nv Conda Environment from the Kernel Selector in VS Code

In [ ]:
# Note that these commands are listed but cannot be executed in the notebook directly.
# Use the kernel selector to activate conda. The next block of code will point to the folder directly
!conda activate SE3nv

### Use `pip` to set up packages

*`cd` command coes not work directly in VS code*

In [ ]:
import os
cur_dir = os.getcwd()
os.chdir('../Tools/RFdiffusion/env/SE3Transformer')

%pip install --no-cache-dir -r requirements.txt
!python setup.py install # Depricated

os.chdir(cur_dir)

### Install RFdiffusion

Does not want to run in VS Code. Can do setup with Conda terminal

In [3]:
os.chdir('../Tools/RFdiffusion')
%pip install -e . # install the rfdiffusion module from the root of the repository

os.chdir(cur_dir)

Obtaining file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of rfdiffusion==1.1.0 from file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for rfdiffusion
Note: you may need to restart the kernel to use updated packages.


## Use RFDiffusion

Once the environment is set up, just use the Kernel picker to use the environment

### Setup

In [51]:
import os, time, subprocess, json

original_directory = os.getcwd()

data_path = "../Data"
rf_diff_path = "../Tools/RFdiffusion/scripts"
protein_mpnn_path = "../Tools/ProteinMPNN/"

pdb_path = os.path.join(data_path, "TIMP3_vs_ADAM17_X_ray.pdb")
output_dir = "../Local/rfdiffusion_output"
pmpnn_out_dir = "../Local/proteinmpnn_output"
output_prefix = "design"


loop_insertion_site = 30
chain_to_design = "B"
fixed_chains = ["A"]
total_length = 121
loop_length = 6
loop_position = 30
contig_string = f"{chain_to_design}1-{loop_position}/{loop_length}-{loop_length+1}/{chain_to_design}{loop_position+loop_length}-{total_length}"
num_sequences_to_generate = 5 

print(original_directory)
print(output_dir)
print(contig_string)

/home/ryangustafson/Documents/GitHub/PhD-Research/Generation
../Local/rfdiffusion_output
B1-30/6-7/B36-121


Debugging

In [32]:
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Run RFdiffusion

In [54]:
if not pdb_path:
    print("Cannot run RFdiffusion without a scaffold PDB file.")
    raise Exception("No PDB File")

print("Preparing to run RFdiffusion...")

# Construct the command for RFdiffusion
run_command = [
    "python",
    os.path.join(rf_diff_path.replace('../', ''), "run_inference.py"),
    f"inference.output_prefix={os.path.join(output_dir.replace('../', ''), output_prefix)}",
    f"inference.input_pdb={pdb_path.replace('../', '')}",
    f'contigmap.contigs=[{contig_string}/0 A219-474]',
    #"denoiser.num_steps=50",
    f"inference.num_designs={num_sequences_to_generate}",
]

print("Running RFdiffusion to generate novel loops and structures...")
print(" ".join(run_command))

# Run the command and stream output live
os.chdir("..") 
st = time.time()
result = subprocess.run(run_command, capture_output=True, text=True)
end = time.time()
os.chdir(original_directory)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

print(f"RFdiffusion finished in {(end-st)/60:.2f} minutes.")

Preparing to run RFdiffusion...
Running RFdiffusion to generate novel loops and structures...
python Tools/RFdiffusion/scripts/run_inference.py inference.output_prefix=Local/rfdiffusion_output/design inference.input_pdb=Data/TIMP3_vs_ADAM17_X_ray.pdb contigmap.contigs=[B1-30/6-7/B36-121/0 A219-474] inference.num_designs=5
--- STDOUT ---
[2025-11-04 16:10:17,893][__main__][INFO] - Found GPU with device_name NVIDIA GeForce RTX 5070 Ti. Will run RFdiffusion on NVIDIA GeForce RTX 5070 Ti
Reading models from /home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models
[2025-11-04 16:10:17,894][rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from /home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models/Base_ckpt.pt
This is inf_conf.ckpt_path
/home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models/Base_ckpt.pt
Assembling -model, -diffuser and -

### ProteinMPNN

In [55]:
os.makedirs(pmpnn_out_dir, exist_ok=True)
mp_seqs = 100

# Create JSONL helper files for ProteinMPNN
chain_json = {f"{output_prefix}_0": [[c for c in fixed_chains], [str(chain_to_design)]]}
with open(f"{pmpnn_out_dir}/{output_prefix}_chain_B.jsonl", "w") as f:
    json.dump(chain_json, f)

# Freeze all residues except the inpainted region
fixed_json = {
    f"{output_prefix}_0": {
        str(chain_to_design): [i for i in list(range(1,31)) + list(range(36,122))]
    }
}
for c in fixed_chains:
    fixed_json[f"{output_prefix}_0"][c] = []
with open(f"{pmpnn_out_dir}/fixed.jsonl", "w") as f:
    json.dump(fixed_json, f)

if not output_dir:
    print("Cannot run ProteinMPNN without a scaffold files.")
    raise Exception("No RFdiffusion Files")

print("Preparing to run ProteinMPNN...")

# Construct the command for RFdiffusion
run_command = [
    "python",
    os.path.join(protein_mpnn_path.replace('../', ''), "protein_mpnn_run.py"),
    "--pdb_path", f"{os.path.join(output_dir.replace('../', ''), output_prefix)}_0.pdb",
    "--out_folder", f"{pmpnn_out_dir.replace('../', '')}",
    "--chain_id_jsonl", f"{pmpnn_out_dir.replace('../', '')}/{output_prefix}_chain_B.jsonl",
    "--fixed_positions_jsonl", f"{pmpnn_out_dir.replace('../', '')}/fixed.jsonl",
    "--num_seq_per_target", f"{mp_seqs}",
    "--sampling_temp", "0.1"
]

print("Running ProteinMPNN to generate sequences...")
print(" ".join(run_command))

# Run the command and stream output live
os.chdir("..") 
st = time.time()
result = subprocess.run(run_command, capture_output=True, text=True)
end = time.time()
os.chdir(original_directory)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

print(f"RFdiffusion finished in {(end-st)/60:.2f} minutes.")

Preparing to run ProteinMPNN...
Running ProteinMPNN to generate sequences...
python Tools/ProteinMPNN/protein_mpnn_run.py --pdb_path Local/rfdiffusion_output/design_0.pdb --out_folder Local/proteinmpnn_output --chain_id_jsonl Local/proteinmpnn_output/design_chain_B.jsonl --fixed_positions_jsonl Local/proteinmpnn_output/fixed.jsonl --num_seq_per_target 100 --sampling_temp 0.1
--- STDOUT ---
----------------------------------------
pssm_jsonl is NOT loaded
----------------------------------------
omit_AA_jsonl is NOT loaded
----------------------------------------
bias_AA_jsonl is NOT loaded
----------------------------------------
tied_positions_jsonl is NOT loaded
----------------------------------------
bias by residue dictionary is not loaded, or not provided
----------------------------------------
----------------------------------------
Number of edges: 48
Training noise level: 0.2A
Generating sequences for: design_0
100 sequences of length 379 generated in 98.5351 seconds

--- ST

### Analyze results

In [72]:
# --- Parse PDB Results to Extract Sequences ---
print("parsing generated PDBs to extract sequences...")

# 3-letter to 1-letter amino acid code map
aa_map = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
        'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
        'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
        'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

generated_sequences = set()
generated_loops = set()
chain_id_to_extract = contig_string.split('/')[0][0] # Get chain from contig
start_res, end_res = map(int, contig_string.split('/')[1].split('-'))
loop_length_range = range(start_res, end_res + 1)

for i in range(num_sequences_to_generate):
    pdb_file_name = f"{output_prefix}_{i}.pdb"
    pdb_file = os.path.join(output_dir, pdb_file_name)
    if os.path.exists(pdb_file):
        current_sequence = []
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith('ATOM') and line[21] == chain_id_to_extract and line[13:15] == "CA":
                    res_name = line[17:20]
                    if res_name in aa_map:
                        current_sequence.append(aa_map[res_name])

        generated_sequences.add("".join(current_sequence))
        loop_seq = "".join(current_sequence[loop_insertion_site:loop_insertion_site + start_res])
        if len(loop_seq) in loop_length_range:
            generated_loops.add(loop_seq)

print(f"\nExtracted {len(generated_sequences)} unique, novel loop sequences.")
print("Here are a few examples (full):")
for seq in list(generated_sequences)[:5]:
    print(f"   - {seq}")
print("Just the loops:")
for seq in list(generated_loops)[:5]:
    print(f"   - {seq}")

parsing generated PDBs to extract sequences...

Extracted 1 unique, novel loop sequences.
Here are a few examples (full):
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKGGGGGGTLVYTIKQMKMYRGFTKMPHVQYIHTEASESLCGLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCN
Just the loops:
   - GGGGGG


In [ ]:
# Final cleanup
original_sequences = set(df['sequence'])
unique_new_sequences = list(generated_sequences - original_sequences)

print(f"\nExtracted {len(unique_new_sequences)} unique, novel loop sequences.")
print("Here are a few examples:")
for seq in unique_new_sequences[:5]:
    print(f"   - {seq}")